# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Dawngend/FlyRank-Machine-Learning-Internship/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

**Card status:** ML-06 was retired upstream on 2026-07-13 and its core folded into ML-07. This is
the optional stretch version, and it is worth doing for one reason: **ML-07 built a hand rule that
weighted staleness at 30 points, the heaviest weight in the scoring function, and never tested
whether staleness predicts anything.** Section 3 runs that test. It should have run first.

Written after ML-07 and ML-08, so it does not pretend to be a prediction. It is the audit that
explains a result those cards had already stumbled into: the rule's heaviest signal fired on 17 of
30,000 rows, and the model ranks it 12th.

Skill loaded: `skills/auditing-signals`. Its two rules do most of the work here: **distributions
before correlations**, and **no verdict from a bucket under the sample-size floor** (50 rows, 30 for
cross-cuts).

## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

Every traffic-like column in this corpus is heavy-tailed, and one of them is heavy-tailed enough to
flip the sign of a correlation. That is not a curiosity, it decides how sections 2 and 3 are allowed
to measure anything.

Read the table below for three things: the **gap between mean and median** (how far the giants drag
the average), the **skew**, and the **zero share** (where a zero might mean "not measured" rather
than "none").

In [1]:
# Section 1 - distributions first, because they decide what tests are legal below.
import sys
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 50)


def find_repo_root() -> Path:
    for base in [Path.cwd(), *Path.cwd().parents, Path("/content/FlyRank-Machine-Learning-Internship")]:
        if (base / "work" / "outputs" / "data_contract.json").exists():
            return base
    raise FileNotFoundError(
        "Could not locate the repo root. In Colab, clone the repo first:\n"
        "  !git clone https://github.com/Dawngend/FlyRank-Machine-Learning-Internship.git"
    )


ROOT = find_repo_root()
frame = pd.read_csv(ROOT / "data" / "processed" / "refresh_feature_vector.csv")
BASE_RATE = frame["is_declining_label"].mean()
FLOOR, CROSS_FLOOR = 50, 30  # sample-size floors from the skill

print(f"rows {len(frame):,}   clients {frame['client_id'].nunique()}   "
      f"base rate (declining) {BASE_RATE:.4f}")

cols = ["impressions_90d", "clicks_90d", "sessions_90d", "ai_sessions_90d",
        "word_count", "search_volume", "avg_position", "ctr"]
dist = pd.DataFrame({
    "mean": frame[cols].mean(),
    "median": frame[cols].median(),
    "p99": frame[cols].quantile(0.99),
    "max": frame[cols].max(),
    "skew": frame[cols].skew(),
    "zero_share": (frame[cols] == 0).mean(),
}).round(3)
print("\n--- distributions ---")
print(dist.to_string())

print("\n--- what this forces ---")
worst = dist["skew"].idxmax()
print(f"most skewed field: {worst} (skew {dist.loc[worst, 'skew']:.1f}, "
      f"mean {dist.loc[worst, 'mean']:,.0f} vs median {dist.loc[worst, 'median']:,.0f})")
for col in ["word_count", "search_volume", "clicks_90d", "ai_sessions_90d"]:
    print(f"  {col:<18} {(frame[col] == 0).mean():>6.1%} zeros")
print("\n-> Pearson correlation on raw values is not usable here. Sections 2 and 3 use")
print("   Spearman, log transforms, or grouped medians, and always show n.")

rows 30,000   clients 32   base rate (declining) 0.5421

--- distributions ---
                     mean   median        p99       max    skew  zero_share
impressions_90d  5200.366   731.00  73505.830  517715.0  11.385       0.000
clicks_90d         16.097     1.00    253.010    4178.0  18.346       0.440
sessions_90d       37.067     7.00    451.010    4345.0  12.127       0.000
ai_sessions_90d     0.204     0.00      5.000      64.0  19.136       0.936
word_count       2310.205  2605.00   7090.080    9546.0   0.492       0.257
search_volume     145.812    10.00   2400.000   74000.0  27.147       0.452
avg_position       16.342    10.80     69.901     245.0   1.984       0.040
ctr                 0.511     0.07      8.330     100.0  17.444       0.440

--- what this forces ---
most skewed field: search_volume (skew 27.1, mean 146 vs median 10)
  word_count          25.7% zeros
  search_volume       45.2% zeros
  clicks_90d          44.0% zeros
  ai_sessions_90d     93.6% zeros

-> Pea

**Read.** `search_volume` has a skew of 27 and 45% zeros. `ai_sessions_90d` is 93.6% zeros.
`clicks_90d` is 44% zeros with a median of 1 against a mean of 16.

**The zero problem is not uniform.** A zero in `clicks_90d` plausibly means zero clicks. A zero in
`word_count` (25.7% of rows) almost certainly means *not measured*, because a published page cannot
have no words. That distinction matters in section 2, where the shortest word-count bucket turns out
to be a measurement artefact rather than a population of thin pages.

## 2. Signal tests, one verdict each

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

Each test states the claim in one sentence, runs on a defined slice, and ends in a verdict with n
visible.

### Signal 1 — "Longer content earns more traffic"

**Verdict: MIXED**, and the interesting part is that the answer depends entirely on how you measure.

Ranked correlation says weakly positive (Spearman 0.108). Raw Pearson says weakly positive (0.082).
**Log-log Pearson says weakly negative (−0.070).** A sign flip between transforms on the same two
columns means neither number describes a real relationship: both are small enough that the giants
decide the sign.

The grouped medians are the honest read, and they do rise with depth: 172 impressions at 1000–2000
words, 997 at 2000–3500, 1340 at 3500+. But the `<1000` bucket has a median of **4 impressions** and
a declining rate of 0.207 against a base of 0.542, which is not a story about thin content winning.
It is the `word_count == 0` rows: pages where length was never measured, sitting alongside pages with
almost no traffic to measure.

**In practice:** depth tracks visibility weakly and directionally in this corpus. It is not a lever
you can pull, and any claim resting on the shortest bucket is resting on missing data.

### Signal 2 — "Fresher content declines less"

**Verdict: OPPOSITE**, weakly, and it is the finding that matters most for the rest of this project.

| Days since update | n | Declining rate |
|---|---:|---:|
| 0–30 | 20,480 | 0.511 |
| 31–90 | 175 | 0.589 |
| 91–180 | 9,171 | **0.611** |
| 181+ | 174 | **0.471** |

Recently updated pages decline *slightly more often* than the base rate would suggest, and the
stalest bucket has the **lowest** declining rate of all four. Spearman between
`days_since_last_update` and the label is **+0.0495**, which is not a relationship, it is a rounding
error with a sign.

Both small buckets clear the 50-row floor, so they can carry a verdict, but only just, and the
verdict they carry is "no usable signal" rather than "staleness protects pages".

**In practice:** editing recency does not tell you which pages are in trouble here. Section 3 shows
what the ML-07 rule did with that fact before anyone had measured it.

### Signal 3 — "High search volume means more impressions"

**Verdict: FALSE** on this slice, replicating the source paper's Myth #2 on a different sample.

Across 16,451 pages with a non-zero volume figure, every correlation is indistinguishable from zero:
Pearson raw 0.003, Pearson log-log −0.002, Spearman 0.001. Median impressions barely move across
buckets (874 → 805 → 838 → 1016) while median search volume moves from 10 to 18,100, a factor of
1,810.

The one thing volume does predict is whether a page beats its own benchmark, and that is arithmetic
rather than signal: 87.6% of pages in the 1–100 bucket exceed their keyword volume, against 6.0% in
the 10K+ bucket. A small denominator is easy to beat.

**In practice:** treat stored search volume as a rough competition hint, never as a traffic forecast
for a specific page.

In [2]:
# Section 2 - the three tests, each ending in a verdict with n visible.


def verdict(name: str, claim: str, table: pd.DataFrame, call: str, practice: str) -> None:
    print(f"\n{'=' * 76}\nSIGNAL: {name}\n  claim: {claim}\n")
    print(table.to_string())
    small = table["n"][table["n"] < FLOOR] if "n" in table else []
    if len(small):
        print(f"\n  NOTE: {len(small)} bucket(s) below the {FLOOR}-row floor - no verdict from those.")
    print(f"\n  VERDICT: {call}\n  in practice: {practice}")


# --- Signal 1: longer content -------------------------------------------------
words = frame.groupby("word_count_tier").agg(
    n=("content_id", "size"),
    median_impressions=("impressions_90d", "median"),
    mean_impressions=("impressions_90d", "mean"),
    declining_rate=("is_declining_label", "mean"),
).round(3)
spear = frame["word_count"].corr(frame["impressions_90d"], method="spearman")
pear = frame["word_count"].corr(frame["impressions_90d"])
loglog = np.log1p(frame["word_count"]).corr(np.log1p(frame["impressions_90d"]))
verdict(
    "longer content earns more traffic",
    "word count and impressions move together",
    words,
    f"MIXED - Spearman {spear:+.3f}, Pearson raw {pear:+.3f}, Pearson log-log {loglog:+.3f}. "
    "A sign flip between transforms means the giants decide the sign, not the relationship.",
    "depth tracks visibility weakly and directionally; the <1000 bucket is word_count==0, "
    f"i.e. not measured ({(frame['word_count'] == 0).mean():.1%} of all rows).",
)

# --- Signal 2: freshness ------------------------------------------------------
fresh = frame.groupby("freshness_tier").agg(
    n=("content_id", "size"),
    declining_rate=("is_declining_label", "mean"),
    median_impressions=("impressions_90d", "median"),
).round(4)
fresh_spear = frame["days_since_last_update"].corr(frame["is_declining_label"], method="spearman")
verdict(
    "fresher content declines less",
    "pages updated recently are less likely to be declining",
    fresh,
    f"OPPOSITE (weakly) - Spearman {fresh_spear:+.4f}. The freshest bucket declines at "
    f"{fresh.loc['0-30', 'declining_rate']:.3f} and the stalest at "
    f"{fresh.loc['181+', 'declining_rate']:.3f}, against a base rate of {BASE_RATE:.3f}.",
    "editing recency does not identify pages in trouble in this corpus.",
)

# --- Signal 3: search volume --------------------------------------------------
nz = frame[frame["search_volume"] > 0].copy()
nz["sv_bucket"] = pd.cut(nz["search_volume"], [0, 100, 1_000, 10_000, np.inf],
                         labels=["1-100", "100-1K", "1K-10K", "10K+"])
volume = nz.groupby("sv_bucket", observed=True).apply(
    lambda d: pd.Series({
        "n": len(d),
        "median_search_volume": d["search_volume"].median(),
        "median_impressions": d["impressions_90d"].median(),
        "share_beating_own_volume": (d["impressions_90d"] > d["search_volume"]).mean(),
    }), include_groups=False).round(4)
verdict(
    "high search volume means more impressions",
    "pages targeting higher-volume keywords earn more impressions",
    volume,
    f"FALSE - Pearson raw {nz['search_volume'].corr(nz['impressions_90d']):+.4f}, "
    f"log-log {np.log1p(nz['search_volume']).corr(np.log1p(nz['impressions_90d'])):+.4f}, "
    f"Spearman {nz['search_volume'].corr(nz['impressions_90d'], method='spearman'):+.4f}. "
    "Median impressions are flat while median volume moves by a factor of ~1,800.",
    "search volume is a competition hint, not a traffic forecast for a page.",
)


SIGNAL: longer content earns more traffic
  claim: word count and impressions move together

                     n  median_impressions  mean_impressions  declining_rate
word_count_tier                                                             
1000-2000         3780               172.0          1233.722           0.556
2000-3500        11263               997.0          5586.166           0.588
3500+             6285              1340.0          7262.685           0.597
<1000              973                 4.0            32.961           0.207
unknown           7699               878.0          5552.992           0.465

  VERDICT: MIXED - Spearman +0.108, Pearson raw +0.082, Pearson log-log -0.070. A sign flip between transforms means the giants decide the sign, not the relationship.
  in practice: depth tracks visibility weakly and directionally; the <1000 bucket is word_count==0, i.e. not measured (25.7% of all rows).

SIGNAL: fresher content declines less
  claim: pages update

## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

### The flag under test

`stale_visible_page`, the **highest-weighted rule in my own ML-07 baseline** at 30 points:

```python
(days_since_last_update >= 180) & (impressions_90d >= 500)
```

The assumption is explicit in the rule's own comment: a page nobody has touched in six months, that
still earns real impressions, is the most urgent refresh case in the corpus.

### The test, and the trap it walks into

The rule fires on **17 rows out of 30,000**. Those 17 rows are declining at **0.941** against a base
rate of 0.542.

That looks like the strongest signal in the entire project. It is not a signal. **n = 17 is below
the 30-row floor for a cross-cut**, and the skill's phrase for exactly this is the right one: a huge
ratio from a tiny cell is noise wearing a costume. Reporting 0.941 as a finding would have been the
single worst claim I could have made in this repo.

### What happens when the cell is big enough to read

Relaxing the staleness cut from 180 days to 90 days, keeping the visibility condition, gives
**n = 6,575** and a declining rate of **0.616** against 0.542. That is a real but modest signal, and
it is worth more than the 0.941 because it can be believed.

And staleness *alone*, with no visibility condition, carries nothing at all: across quartiles of
`days_since_last_update` the declining rate is 0.539, 0.546, 0.547. Flat.

### Verdict: **MIXED, and the rule's specific cut is FALSE**

- Staleness alone: **no signal** (flat across quartiles).
- Staleness combined with visibility, at a usable threshold: **weak CONFIRMED** (0.616 vs 0.542).
- The rule's actual 180-day threshold: **insufficient data to support any verdict**, and the 30
  points it assigns are unearned.

### What this predicted, before either card was written

Two later results agree with this section, independently:

- **ML-07** measured `stale_visible_page` at precision 0.529, *below* the 0.542 base rate, on its 17
  rows.
- **ML-08** gave the random forest free access to `days_since_last_update` and it ranked the feature
  **12th at 0.023 importance**, below scroll rate.

Three separate methods, one conclusion: **recency of editing does not carry information about decline
in this corpus.** That is a negative result, and it is the most useful thing this notebook produced.

**What I would change in ML-07 given this:** drop the 30-point staleness weight to zero, or rebuild
it at a 90-day cut where the cell is large enough to measure. I have left the rule as it was, because
ML-08 and ML-09 are both evaluated against it and changing the baseline after the fact would
invalidate that comparison.

In [3]:
# Section 3 - test the assumption behind the rule's heaviest weight.
stale_visible = (frame["days_since_last_update"] >= 180) & (frame["impressions_90d"] >= 500)
n_fires = int(stale_visible.sum())
rate = frame.loc[stale_visible, "is_declining_label"].mean()

print("--- the rule as written: days_since_last_update >= 180 AND impressions_90d >= 500 ---")
print(f"fires on            : {n_fires} of {len(frame):,} rows ({stale_visible.mean():.4%})")
print(f"declining rate      : {rate:.4f}")
print(f"base rate           : {BASE_RATE:.4f}")
print(f"looks like a lift of: {rate / BASE_RATE:.2f}x")
if n_fires < CROSS_FLOOR:
    print(f"\n  *** n = {n_fires} is BELOW the {CROSS_FLOOR}-row cross-cut floor. ***")
    print("  No verdict may be drawn from this cell. A huge ratio from a tiny cell is noise")
    print("  wearing a costume, and 0.94 from 17 rows is exactly that.")

print("\n--- relaxing the staleness cut until the cell is readable ---")
rows = []
for days in (30, 60, 90, 120, 180):
    m = (frame["days_since_last_update"] >= days) & (frame["impressions_90d"] >= 500)
    rows.append({
        "staleness_cut_days": days,
        "n": int(m.sum()),
        "declining_rate": round(float(frame.loc[m, "is_declining_label"].mean()), 4) if m.any() else None,
        "lift_vs_base": round(float(frame.loc[m, "is_declining_label"].mean() / BASE_RATE), 3) if m.any() else None,
        "above_floor": bool(m.sum() >= CROSS_FLOOR),
    })
print(pd.DataFrame(rows).to_string(index=False))

print("\n--- staleness alone, no visibility condition, by quartile ---")
quartiles = frame.assign(q=pd.qcut(frame["days_since_last_update"], 4, duplicates="drop"))
alone = quartiles.groupby("q", observed=True).agg(
    n=("content_id", "size"), declining_rate=("is_declining_label", "mean")).round(4)
print(alone.to_string())
spread = alone["declining_rate"].max() - alone["declining_rate"].min()
print(f"\nspread across quartiles: {spread:.4f}")
print("-> flat. Staleness on its own separates nothing.")

print("\n--- three methods, one conclusion ---")
import json  # noqa: E402
rule_metrics = json.loads(
    (ROOT / "work" / "outputs" / "baseline_action_score_metrics.json").read_text(encoding="utf-8"))
model_metrics = json.loads(
    (ROOT / "work" / "outputs" / "model_comparison.json").read_text(encoding="utf-8"))
importances = model_metrics["random_forest_top_features"]
ranked = sorted(importances.items(), key=lambda kv: -kv[1])
position = [i for i, (k, _) in enumerate(ranked, 1) if "days_since_last_update" in k]

print(f"  ML-06 (here)  : staleness quartile spread {spread:.4f} -> no signal")
print(f"  ML-07 (rule)  : stale_visible_page precision "
      f"{rule_metrics['precision_by_reason']['stale_visible_page']:.4f} vs base {BASE_RATE:.4f} "
      "-> below chance")
if position:
    name, value = ranked[position[0] - 1]
    print(f"  ML-08 (model) : days_since_last_update ranked #{position[0]} at {value:.4f} importance")
print("\n  -> recency of editing does not carry information about decline in this corpus.")

--- the rule as written: days_since_last_update >= 180 AND impressions_90d >= 500 ---
fires on            : 17 of 30,000 rows (0.0567%)
declining rate      : 0.9412
base rate           : 0.5421
looks like a lift of: 1.74x

  *** n = 17 is BELOW the 30-row cross-cut floor. ***
  No verdict may be drawn from this cell. A huge ratio from a tiny cell is noise
  wearing a costume, and 0.94 from 17 rows is exactly that.

--- relaxing the staleness cut until the cell is readable ---
 staleness_cut_days    n  declining_rate  lift_vs_base  above_floor
                 30 6663          0.6152         1.135         True
                 60 6591          0.6166         1.137         True
                 90 6575          0.6164         1.137         True
                120   22          0.7273         1.342        False
                180   17          0.9412         1.736        False

--- staleness alone, no visibility condition, by quartile ---
                    n  declining_rate
q         

## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

**Stop prioritising refreshes by edit date.** In this corpus, how long ago a page was last touched
tells you essentially nothing about whether it is declining: the declining rate is flat across every
quartile of edit recency, and the freshest pages decline slightly more often than the stalest ones.
A refresh calendar built on "nothing has touched this in six months" will spend its hours on pages
chosen at close to random.

**Prioritise by sustained visibility instead.** The signal that survives every test in this project
is whether a page shows up consistently in search and how old the content itself is, not when someone
last edited it. That is what the queue in ML-10 ranks on, and it is why the readable hand rule can
only explain about half of what the model selects.

**And check the size of the cell before believing the number.** The most impressive-looking result in
this notebook, a 94% declining rate on the highest-weighted rule in my own baseline, came from 17
rows and means nothing. That one is worth carrying past this dataset.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

Every verdict above has a table with visible n's under it, and the one cell that fell below the
sample-size floor is reported as insufficient rather than as a finding.